# simulations_v073_auxiliary

This notebook contains the reusable backend for the paired clean notebook, including v062 translation checks.

## Imports

In [ ]:
from collections import OrderedDict
import copy
from pathlib import Path
import os
import re
import sys
from textwrap import dedent

import numpy as np

_AUXILIARY_DIR = Path(AUXILIARY_NOTEBOOK).resolve().parent if 'AUXILIARY_NOTEBOOK' in globals() else Path.cwd()
_VENDOR_AUTO_CANDIDATES = [
    _AUXILIARY_DIR / '_auto_vendor_test',
    _AUXILIARY_DIR / 'Legacy Scripts' / '_auto_vendor_test',
    Path.cwd() / '_auto_vendor_test',
    Path.cwd() / 'Legacy Scripts' / '_auto_vendor_test',
    Path('/Users/tristan/Documents/diagnostics/Legacy Scripts/_auto_vendor_test'),
    Path('/Users/tristan/Documents/diagnostics/Legacy Scripts/_auto_vendor'),
]
for _vendor_auto in _VENDOR_AUTO_CANDIDATES:
    if _vendor_auto.exists():
        vendor_path = str(_vendor_auto)
        if vendor_path not in sys.path:
            sys.path.insert(0, vendor_path)
        break

from auto import AUTOCommands as ac
from auto import runAUTO as ra


## Model Parsing and Evaluation

In [ ]:
SAFE_GLOBALS = {"__builtins__": {}}
NUMERIC_ENV = {
    'exp': np.exp,
    'log': np.log,
    'sqrt': np.sqrt,
    'sin': np.sin,
    'cos': np.cos,
    'tan': np.tan,
    'sinh': np.sinh,
    'cosh': np.cosh,
    'tanh': np.tanh,
    'abs': np.abs,
    'pi': np.pi,
}


def strip_comment(line):
    return line.split('#', 1)[0].rstrip()


def collect_logical_lines(text):
    logical = []
    buffer = ''
    for raw in text.splitlines():
        line = strip_comment(raw).strip()
        if not line:
            continue
        if line.endswith('\\'):
            buffer += line[:-1].strip() + ' '
            continue
        line = (buffer + line).strip()
        buffer = ''
        if line:
            logical.append(line)
    if buffer:
        logical.append(buffer.strip())
    return logical


def parse_assignments(block):
    result = OrderedDict()
    for item in block.split(','):
        piece = item.strip()
        if not piece:
            continue
        name, value = piece.split('=', 1)
        result[name.strip()] = value.strip()
    return result


def parse_function_definition(line):
    match = re.match(r'^([A-Za-z_]\w*)\((.*)\)\s*=\s*(.+)$', line)
    if match is None:
        return None
    name, args_text, expr = match.groups()
    args = [arg.strip() for arg in args_text.split(',') if arg.strip()]
    return name, args, expr.strip()


def parse_equation_definition(line):
    match = re.match(r"^([A-Za-z_]\w*)'\s*=\s*(.+)$", line)
    if match is None:
        return None
    name, expr = match.groups()
    return name, expr.strip()


def parse_model_text(ode_text):
    logical_lines = collect_logical_lines(ode_text)

    params = OrderedDict()
    inits = OrderedDict()
    auto_settings = OrderedDict()
    function_defs = []
    alias_defs = []
    equation_defs = OrderedDict()

    for line in logical_lines:
        if line.startswith('par '):
            params.update(parse_assignments(line[4:]))
            continue
        if line.startswith('init '):
            inits.update(parse_assignments(line[5:]))
            continue
        if line.startswith('@ '):
            auto_settings.update(parse_assignments(line[2:]))
            continue

        equation = parse_equation_definition(line)
        if equation is not None:
            equation_defs[equation[0]] = equation[1]
            continue

        function_def = parse_function_definition(line)
        if function_def is not None:
            function_defs.append(function_def)
            continue

        if '=' in line:
            name, expr = line.split('=', 1)
            alias_defs.append((name.strip(), expr.strip()))
            continue

        raise ValueError(f"Could not parse line: {line}")

    state_names = list(inits.keys())
    param_names = list(params.keys())

    if set(equation_defs.keys()) != set(state_names):
        raise ValueError('State equations and initial-condition variables do not match.')

    return {
        'params': params,
        'inits': inits,
        'auto_settings': auto_settings,
        'function_defs': function_defs,
        'alias_defs': alias_defs,
        'equation_defs': equation_defs,
        'state_names': state_names,
        'param_names': param_names,
        'param_values': OrderedDict((name, float(value)) for name, value in params.items()),
        'init_values': OrderedDict((name, float(value)) for name, value in inits.items()),
    }


def make_model_env(model, state_dict, param_dict):
    env = dict(NUMERIC_ENV)
    env.update(param_dict)
    env.update(state_dict)

    def build_helper(arg_names, expr):
        def helper(*values):
            local = dict(env)
            local.update(dict(zip(arg_names, values)))
            return eval(expr, SAFE_GLOBALS, local)
        return helper

    for name, arg_names, expr in model['function_defs']:
        env[name] = build_helper(arg_names, expr)

    for name, expr in model['alias_defs']:
        env[name] = eval(expr, SAFE_GLOBALS, env)

    return env


def full_rhs(model, y, p):
    state_dict = OrderedDict(zip(model['state_names'], np.asarray(y, dtype=float)))
    env = make_model_env(model, state_dict, p)
    return np.array(
        [eval(model['equation_defs'][name], SAFE_GLOBALS, env) for name in model['state_names']],
        dtype=float,
    )


def full_jacobian(model, y, p, h=1e-7):
    y = np.asarray(y, dtype=float)
    J = np.zeros((len(y), len(y)), dtype=float)
    for idx in range(len(y)):
        plus = y.copy()
        minus = y.copy()
        plus[idx] += h
        minus[idx] -= h
        J[:, idx] = (full_rhs(model, plus, p) - full_rhs(model, minus, p)) / (2.0 * h)
    return J


def numerical_jacobian(func, x, h=1e-7):
    x = np.asarray(x, dtype=float)
    base = np.asarray(func(x), dtype=float)
    J = np.zeros((len(base), len(x)), dtype=float)
    for idx in range(len(x)):
        plus = x.copy()
        minus = x.copy()
        plus[idx] += h
        minus[idx] -= h
        J[:, idx] = (np.asarray(func(plus), dtype=float) - np.asarray(func(minus), dtype=float)) / (2.0 * h)
    return J


## Experiment Preparation, Continuation, and Plot Configuration

In [ ]:

DEFAULT_EXPERIMENT_CONFIG = {
    'plot_state': 'PL',
    'boundary_free_variables': ['FL', 'FP'],
    'boundary_fixed_values': {'PL': 0.0, 'JL': 0.0, 'PP': 0.0, 'JP': 0.0},
    'invasion_variables': ['PL', 'JL', 'PP', 'JP'],
    'zero_branch_variables': ['PL', 'PP'],
    'positive_branch_variables': ['PL', 'PP'],
    'branch_point_label': 'BP1',
    'fold_label': 'LP1',
    'primary_min': None,
    'primary_max': None,
    'eq_ds': None,
    'eq_dsmin': None,
    'eq_dsmax': None,
    'eq_norm_min': 0.0,
    'eq_norm_max': 100.0,
    'one_d_plot_min': -0.5,
    'one_d_plot_max': None,
    'ecology_plot_tol': 1.0e-8,
    'show_zero_branch': True,
    'show_zero_branch_unstable': False,
    'show_other_branches': False,
    'positive_branch_force_solid': True,
    'positive_branch_color': 'red',
    'zero_branch_color': 'black',
    'other_branch_color': '0.55',
    'positive_branch_linewidth': 1.8,
    'zero_branch_linewidth': 1.6,
    'other_branch_linewidth': 1.3,
    'one_d_plot_grid': False,
    'branch_zero_tol': 1.0e-7,
    'branch_nonnegative_tol': 1.0e-7,
    'branch_min_points': 5,
    'stability_tol': 1.0e-7,
    'rescue_step': 0.005,
    'rescue_primary_stop': 0.01,
    'rescue_positive_tol': 1.0e-8,
    'rescue_monotonic_tol': 1.0e-8,
    'rescue_max_it': 30,
    'rescue_newton_tol': 1.0e-12,
    'bistability_plot_min': 0.0,
    'bistability_plot_max': None,
    'bistability_plot_ymin': -0.02,
    'bistability_plot_grid': False,
    'stable_branch_color': 'black',
    'unstable_branch_color': 'black',
    'stable_branch_linewidth': 2.6,
    'unstable_branch_linewidth': 2.1,
    'eq_nmx': 4000,
    'eq_npr': 200,
    'codim2_ds': 1.0e-3,
    'codim2_dsmin': 1.0e-5,
    'codim2_dsmax': 5.0e-3,
    'codim2_nmx': 8000,
    'codim2_npr': 400,
    'two_d_plot_ymin': None,
    'two_d_plot_ymax': None,
}


REQUIRED_CONFIG_KEYS = (
    'model_name',
    'primary_continuation_parameter',
    'secondary_continuation_parameter',
    'secondary_min',
    'secondary_max',
)


def normalize_experiment_config(config):
    missing = [key for key in REQUIRED_CONFIG_KEYS if key not in config]
    if missing:
        joined = ', '.join(missing)
        raise KeyError(f'Missing required experiment config keys: {joined}')

    resolved = copy.deepcopy(DEFAULT_EXPERIMENT_CONFIG)
    resolved.update(config)
    return resolved

def validate_experiment_config(model, config):
    if config['primary_continuation_parameter'] not in model['param_names']:
        raise ValueError('Primary continuation parameter is not in the model parameter list.')
    if config['secondary_continuation_parameter'] not in model['param_names']:
        raise ValueError('Secondary continuation parameter is not in the model parameter list.')
    if config['plot_state'] not in model['state_names']:
        raise ValueError('Plot state is not in the model state list.')
    for name in config['boundary_free_variables']:
        if name not in model['state_names']:
            raise ValueError(f'Unknown boundary free variable: {name}')
    for name in config['boundary_fixed_values']:
        if name not in model['state_names']:
            raise ValueError(f'Unknown boundary fixed variable: {name}')
    for group in ['invasion_variables', 'zero_branch_variables', 'positive_branch_variables']:
        for name in config[group]:
            if name not in model['state_names']:
                raise ValueError(f'Unknown state in {group}: {name}')


def build_boundary_state(model, config, free_values):
    state_index = {name: idx for idx, name in enumerate(model['state_names'])}
    state = np.zeros(len(model['state_names']), dtype=float)
    for name, value in config['boundary_fixed_values'].items():
        state[state_index[name]] = float(value)
    for name, value in zip(config['boundary_free_variables'], free_values):
        state[state_index[name]] = float(value)
    return state


def boundary_rhs(model, config, free_values, p):
    state = build_boundary_state(model, config, free_values)
    free_index = [model['state_names'].index(name) for name in config['boundary_free_variables']]
    return full_rhs(model, state, p)[free_index]


def solve_boundary_equilibrium(model, config, p, tol=1e-12, max_iter=40):
    x = np.array([model['init_values'][name] for name in config['boundary_free_variables']], dtype=float)

    for _ in range(max_iter):
        f = boundary_rhs(model, config, x, p)
        if np.linalg.norm(f, ord=np.inf) < tol:
            return build_boundary_state(model, config, x)

        J = numerical_jacobian(lambda z: boundary_rhs(model, config, z, p), x)
        step = np.linalg.solve(J, -f)
        current_norm = np.linalg.norm(f, ord=np.inf)
        step_scale = 1.0

        while step_scale > 1e-8:
            candidate = x + step_scale * step
            if np.all(candidate > 0.0):
                candidate_norm = np.linalg.norm(boundary_rhs(model, config, candidate, p), ord=np.inf)
                if candidate_norm < current_norm:
                    x = candidate
                    break
            step_scale *= 0.5
        else:
            x = np.maximum(x + 0.1 * step, 1e-10)

    raise RuntimeError('Failed to converge to the boundary equilibrium.')


def invasion_jacobian(model, config, primary_value, boundary_state, p, h=1e-7):
    params = dict(p)
    params[config['primary_continuation_parameter']] = float(primary_value)
    invasion_index = [model['state_names'].index(name) for name in config['invasion_variables']]
    base = np.asarray(boundary_state, dtype=float)
    J = np.zeros((len(invasion_index), len(invasion_index)), dtype=float)
    for col, idx in enumerate(invasion_index):
        plus = base.copy()
        minus = base.copy()
        plus[idx] += h
        minus[idx] -= h
        J[:, col] = (
            full_rhs(model, plus, params)[invasion_index] - full_rhs(model, minus, params)[invasion_index]
        ) / (2.0 * h)
    return J


def boundary_det(model, config, primary_value, boundary_state, p):
    return float(np.linalg.det(invasion_jacobian(model, config, primary_value, boundary_state, p)))


def find_linear_thresholds(model, config, boundary_state, p, lo, hi, n_samples=600, tol=1e-11):
    grid = np.linspace(lo, hi, n_samples)
    det_values = np.array([boundary_det(model, config, value, boundary_state, p) for value in grid], dtype=float)
    roots = []

    for left_value, right_value, left_det, right_det in zip(grid[:-1], grid[1:], det_values[:-1], det_values[1:]):
        if left_det == 0.0:
            roots.append(float(left_value))
            continue
        if left_det * right_det > 0.0:
            continue

        a = float(left_value)
        b = float(right_value)
        fa = float(left_det)

        for _ in range(80):
            m = 0.5 * (a + b)
            fm = boundary_det(model, config, m, boundary_state, p)
            if abs(fm) < tol or abs(b - a) < tol:
                a = m
                b = m
                break
            if fa * fm <= 0.0:
                b = m
            else:
                a = m
                fa = fm

        roots.append(0.5 * (a + b))

    deduped = []
    for root in roots:
        if not deduped or abs(root - deduped[-1]) > 1e-6:
            deduped.append(root)

    return deduped


def fmt_fortran(value):
    return f"{float(value):.16e}".replace('e', 'D')


def chunked(items, size):
    return [items[idx:idx + size] for idx in range(0, len(items), size)]


def wrap_fortran_assignment(lhs, expr, indent='  ', width=100):
    statement = f"{lhs} = {expr}"
    tokens = dedent(statement).strip().split()
    lines = []
    current = ''
    for token in tokens:
        candidate = token if not current else current + ' ' + token
        limit = width - len(indent) if not lines else width - len(indent) - 4
        if len(candidate) <= limit:
            current = candidate
        else:
            lines.append(current)
            current = token
    if current:
        lines.append(current)
    if len(lines) == 1:
        return indent + lines[0]
    output = [indent + lines[0] + ' &']
    for part in lines[1:-1]:
        output.append(indent + '  & ' + part + ' &')
    output.append(indent + '  & ' + lines[-1])
    return '\n'.join(output)


def generate_auto_files(model, config, boundary_state, workdir, model_name, primary_min, primary_max):
    param_names = model['param_names']
    state_names = model['state_names']
    param_values = model['param_values']
    alias_defs = model['alias_defs']
    function_defs = model['function_defs']
    equation_defs = model['equation_defs']

    NPAR = max(40, len(param_names))
    param_index = {name: idx for idx, name in enumerate(param_names, start=1)}
    state_index = {name: idx for idx, name in enumerate(state_names, start=1)}
    primary_index = param_index[config['primary_continuation_parameter']]

    decl_lines = []
    for group in [state_names, param_names, [name for name, _ in alias_defs]]:
        if not group:
            continue
        for part in chunked(group, 8):
            decl_lines.append("  DOUBLE PRECISION :: " + ",".join(part))

    state_assignments = [f"  {name}=U({state_index[name]})" for name in state_names]
    param_assignments = [f"  {name}=PAR({param_index[name]})" for name in param_names]
    alias_assignments = [wrap_fortran_assignment(name, expr) for name, expr in alias_defs]
    f_assignments = [wrap_fortran_assignment(f"F({state_index[name]})", equation_defs[name]) for name in state_names]

    internal_functions = []
    for name, args, expr in function_defs:
        lines = [f"  DOUBLE PRECISION FUNCTION {name}({','.join(args)})", "    IMPLICIT NONE"]
        if args:
            for part in chunked(args, 8):
                lines.append("    DOUBLE PRECISION, INTENT(IN) :: " + ",".join(part))
        lines.append(wrap_fortran_assignment(name, expr, indent='    '))
        lines.append(f"  END FUNCTION {name}")
        internal_functions.append('\n'.join(lines))

    stpnt_lines = [f"  PAR(1:{NPAR})=0.D0", "  U(1:NDIM)=0.D0"]
    for name in state_names:
        stpnt_lines.append(f"  U({state_index[name]})={fmt_fortran(boundary_state[state_names.index(name)])}")
    for name in param_names:
        stpnt_lines.append(f"  PAR({param_index[name]})={fmt_fortran(param_values[name])}")
    stpnt_block = '\n'.join(stpnt_lines)

    func_lines = [
        "SUBROUTINE FUNC(NDIM,U,ICP,PAR,IJAC,F,DFDU,DFDP)",
        "  IMPLICIT NONE",
        "  INTEGER, INTENT(IN) :: NDIM, IJAC",
        "  INTEGER, INTENT(IN) :: ICP(*)",
        "  DOUBLE PRECISION, INTENT(IN) :: U(NDIM)",
        "  DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)",
        "  DOUBLE PRECISION, INTENT(OUT) :: F(NDIM)",
        "  DOUBLE PRECISION, INTENT(INOUT) :: DFDU(NDIM,*), DFDP(NDIM,*)",
    ]
    func_lines.extend(decl_lines)
    func_lines.extend(state_assignments)
    func_lines.extend(param_assignments)
    func_lines.extend(alias_assignments)
    func_lines.extend(f_assignments)
    func_lines.append("  RETURN")
    func_lines.append("CONTAINS")
    func_lines.extend(internal_functions)
    func_lines.append("END SUBROUTINE FUNC")
    func_text = '\n'.join(func_lines)

    stpnt_text = dedent(f'''
    SUBROUTINE STPNT(NDIM,U,PAR,T)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM
      DOUBLE PRECISION, INTENT(OUT) :: U(NDIM)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
      DOUBLE PRECISION, INTENT(IN) :: T

    {stpnt_block}
    END SUBROUTINE STPNT
    ''').strip()

    bcnd_text = dedent('''
    SUBROUTINE BCND(NDIM,PAR,ICP,NBC,U0,U1,FB,IJAC,DBC)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM,NBC,IJAC
      INTEGER, INTENT(IN) :: ICP(*)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
      DOUBLE PRECISION, INTENT(IN) :: U0(NDIM),U1(NDIM)
      DOUBLE PRECISION, INTENT(OUT) :: FB(*)
      DOUBLE PRECISION, INTENT(INOUT) :: DBC(*)
    END SUBROUTINE BCND

    SUBROUTINE ICND(NDIM,PAR,ICP,NINT,U,UOLD,UDOT,UPOLD,FI,IJAC,DINT)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM,NINT,IJAC
      INTEGER, INTENT(IN) :: ICP(*)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
      DOUBLE PRECISION, INTENT(IN) :: U(NDIM),UOLD(NDIM),UDOT(NDIM),UPOLD(NDIM)
      DOUBLE PRECISION, INTENT(OUT) :: FI(*)
      DOUBLE PRECISION, INTENT(INOUT) :: DINT(*)
    END SUBROUTINE ICND

    SUBROUTINE FOPT(NDIM,U,ICP,PAR,IJAC,FS,DFDU,DFDP)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM,IJAC
      INTEGER, INTENT(IN) :: ICP(*)
      DOUBLE PRECISION, INTENT(IN) :: U(NDIM)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
      DOUBLE PRECISION, INTENT(OUT) :: FS
      DOUBLE PRECISION, INTENT(INOUT) :: DFDU(*),DFDP(*)
    END SUBROUTINE FOPT

    SUBROUTINE PVLS(NDIM,U,PAR)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM
      DOUBLE PRECISION, INTENT(IN) :: U(NDIM)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
    END SUBROUTINE PVLS
    ''').strip()

    f90_text = func_text + '\n\n' + stpnt_text + '\n\n' + bcnd_text + '\n'

    parnames_text = ', '.join(f"{idx}: '{name}'" for idx, name in enumerate(param_names, start=1))
    unames_text = ', '.join(f"{idx}: '{name}'" for idx, name in enumerate(state_names, start=1))

    c_text = dedent(f'''
    e = '{model_name}'
    NDIM = {len(state_names)}
    NPAR = {NPAR}
    IPS = 1
    IRS = 0
    ILP = 1
    ICP = [{primary_index}]
    NTST = 20
    NCOL = 4
    IAD = 3
    ISP = 2
    ISW = 1
    IPLT = 0
    NBC = 0
    NINT = 0
    NMX = 1000
    NPR = 50
    MXBF = 10
    IID = 2
    ITMX = 10
    ITNW = 9
    NWTN = 7
    JAC = 0
    EPSL = 1e-8
    EPSU = 1e-8
    EPSS = 1e-6
    DS = 1e-4
    DSMIN = 1e-6
    DSMAX = 2e-2
    IADS = 1
    RL0 = {primary_min}
    RL1 = {primary_max}
    A0 = 0.0
    A1 = 20.0
    UZR = {{}}
    UZSTOP = {{{primary_index}: [{primary_min}, {primary_max}]}}
    SP = ['BP', 'LP', 'HB']
    STOP = []
    parnames = {{{parnames_text}}}
    unames = {{{unames_text}}}
    ''').strip() + "\n"

    Path(workdir / f"{model_name}.f90").write_text(f90_text)
    Path(workdir / f"c.{model_name}").write_text(c_text)


def prepare_experiment(config, ode_text):
    config = normalize_experiment_config(config)
    model_name = config['model_name']
    workdir = Path.cwd() / f"{model_name}_auto"
    workdir.mkdir(parents=True, exist_ok=True)
    os.chdir(workdir)

    Path(f"{model_name}.ode").write_text(ode_text)

    model = parse_model_text(ode_text)
    validate_experiment_config(model, config)

    primary_min = (
        float(model['auto_settings'].get('PARMIN', -0.25))
        if config['primary_min'] is None
        else float(config['primary_min'])
    )
    primary_max = (
        float(model['auto_settings'].get('PARMAX', 0.5))
        if config['primary_max'] is None
        else float(config['primary_max'])
    )

    boundary_state = solve_boundary_equilibrium(model, config, model['param_values'])
    boundary_residual = np.linalg.norm(full_rhs(model, boundary_state, model['param_values']), ord=np.inf)
    linear_thresholds = find_linear_thresholds(
        model,
        config,
        boundary_state,
        model['param_values'],
        primary_min,
        primary_max,
    )

    generate_auto_files(model, config, boundary_state, workdir, model_name, primary_min, primary_max)

    return {
        'config': config,
        'model_name': model_name,
        'workdir': workdir,
        'model': model,
        'primary_min': primary_min,
        'primary_max': primary_max,
        'boundary_state': boundary_state,
        'boundary_residual': boundary_residual,
        'linear_thresholds': linear_thresholds,
        'runner': None,
        'one_d_plot_filename': f"{model_name}_1d_{config['plot_state'].lower()}_vs_{config['primary_continuation_parameter']}.png",
        'two_d_plot_filename': f"{model_name}_2d_{config['secondary_continuation_parameter']}_vs_{config['primary_continuation_parameter']}.png",
    }


def show_experiment_summary(experiment):
    model = experiment['model']
    config = experiment['config']
    print(f"Notebook model name: {experiment['model_name']}")
    print(f"Working directory: {experiment['workdir']}")
    print('Boundary equilibrium:')
    for name, value in zip(model['state_names'], experiment['boundary_state']):
        print(f"  {name}* = {value:.12f}")
    print(f"Boundary residual: {experiment['boundary_residual']:.3e}")
    print('Predicted thresholds:')
    for idx, value in enumerate(experiment['linear_thresholds'], start=1):
        print(f"  root {idx}: {config['primary_continuation_parameter']} = {value:.12f}")


def ensure_runner(experiment):
    if experiment['runner'] is None:
        experiment['runner'] = ra.runAUTO()
    return experiment['runner']


def run_one_d_continuation(experiment):
    runner = ensure_runner(experiment)
    config = experiment['config']
    model_name = experiment['model_name']

    eq_forward = ac.run(
        e=model_name,
        c=model_name,
        runner=runner,
        NMX=config['eq_nmx'],
        NPR=config['eq_npr'],
    )
    eq_backward = ac.run(
        DS='-',
        runner=runner,
        NMX=config['eq_nmx'],
        NPR=config['eq_npr'],
    )
    eq_diagram = (eq_forward + eq_backward).relabel()
    ac.save(eq_diagram, 'eq')

    experiment['eq_diagram'] = eq_diagram
    experiment['bp_seed'] = eq_diagram(config['branch_point_label'])
    experiment['lp_seed'] = eq_diagram(config['fold_label'])
    return eq_diagram


def continue_locus(experiment, seed, save_name):
    runner = ensure_runner(experiment)
    config = experiment['config']
    locus = ac.run(
        seed,
        runner=runner,
        ICP=[config['primary_continuation_parameter'], config['secondary_continuation_parameter']],
        ISW=2,
        DS=config['codim2_ds'],
        DSMIN=config['codim2_dsmin'],
        DSMAX=config['codim2_dsmax'],
        NMX=config['codim2_nmx'],
        NPR=config['codim2_npr'],
        UZSTOP={
            config['primary_continuation_parameter']: [experiment['primary_min'], experiment['primary_max']],
            config['secondary_continuation_parameter']: [config['secondary_min'], config['secondary_max']],
        },
    ).relabel()
    ac.save(locus, save_name)
    return locus


def run_two_d_continuation(experiment):
    if 'bp_seed' not in experiment or 'lp_seed' not in experiment:
        raise RuntimeError('Run the one-dimensional continuation first.')
    bp_locus = continue_locus(experiment, experiment['bp_seed'], 'bp_curve')
    lp_locus = continue_locus(experiment, experiment['lp_seed'], 'lp_curve')
    codim2 = (bp_locus + lp_locus).relabel()
    ac.save(codim2, 'codim2')
    experiment['bp_locus'] = bp_locus
    experiment['lp_locus'] = lp_locus
    experiment['codim2'] = codim2
    return codim2


def get_matplotlib_pyplot():
    try:
        import matplotlib.pyplot as plt
    except ImportError as exc:
        raise RuntimeError('matplotlib is required for headless AUTO plotting.') from exc
    return plt


class HeadlessAutoPlotter:
    def __init__(self, name=None, templates=None, **kw):
        self.diagram = ac.loadbd(name) if isinstance(name, str) else name
        self.options = {
            'stability': False,
            'grid': False,
            'bifurcation_x': [0],
            'bifurcation_y': [1],
            'xlabel': '',
            'ylabel': '',
            'title': '',
            'xlim': None,
            'ylim': None,
        }
        if kw:
            self.config(**kw)


    def config(self, **kw):
        self.options.update(kw)
        return self


    def _branches(self):
        return list(getattr(self.diagram, 'branches', self.diagram))


    def _coordnames(self):
        for branch in self._branches():
            coordnames = getattr(branch, 'coordnames', None)
            if coordnames:
                return list(coordnames)
        return []


    def _resolve_columns(self, spec):
        coordnames = self._coordnames()
        columns = spec
        if columns is None:
            return []
        if not isinstance(columns, (list, tuple)):
            columns = [columns]
        resolved = []
        for item in columns:
            if isinstance(item, int):
                resolved.append(item)
            else:
                resolved.append(coordnames.index(str(item)))
        return resolved


    def savefig(self, filename):
        plt = get_matplotlib_pyplot()
        xcolumns = self._resolve_columns(self.options.get('bifurcation_x'))
        ycolumns = self._resolve_columns(self.options.get('bifurcation_y'))
        branches = self._branches()
        fig, ax = plt.subplots(figsize=(6.4, 4.8))
        colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', ['C0'])

        for branch_index, branch in enumerate(branches):
            color = colors[branch_index % len(colors)]
            for xcolumn, ycolumn in zip(xcolumns, ycolumns):
                x = np.asarray(branch.coordarray[xcolumn], dtype=float)
                y = np.asarray(branch.coordarray[ycolumn], dtype=float)

                if self.options.get('stability', False):
                    old = 0
                    stability = branch.stability()
                    if not stability:
                        stability = [len(x)]
                    for point in stability:
                        abs_point = abs(int(point))
                        if abs_point > 1 or point == stability[-1]:
                            xs = x[old:abs_point]
                            ys = y[old:abs_point]
                            if len(xs) >= 2:
                                ax.plot(
                                    xs,
                                    ys,
                                    color=color,
                                    linestyle='-' if point < 0 else '--',
                                    linewidth=1.6,
                                )
                            old = max(abs_point - 1, 0)
                else:
                    if len(x) >= 2:
                        ax.plot(x, y, color=color, linewidth=1.6)

        ax.set_xlabel(self.options.get('xlabel', ''))
        ax.set_ylabel(self.options.get('ylabel', ''))
        ax.set_title(self.options.get('title', ''))
        xlim = self.options.get('xlim')
        ylim = self.options.get('ylim')
        if xlim is not None:
            ax.set_xlim(*xlim)
        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.grid(bool(self.options.get('grid', False)))
        fig.tight_layout()
        fig.savefig(filename, dpi=150, bbox_inches='tight')
        plt.close(fig)


_AUTO_PLOT = ac.plot


def patched_auto_plot(name=None, templates=None, **kw):
    plotter = _AUTO_PLOT(name, templates, **kw)
    if plotter is not None:
        return plotter
    return HeadlessAutoPlotter(name, templates=templates, **kw)


ac.plot = patched_auto_plot


def configure_auto_1d_plot(plotter, experiment, stability=True, grid=False):
    config = experiment['config']
    plotter.config(
        stability=stability,
        grid=grid,
        bifurcation_x=[config['primary_continuation_parameter']],
        bifurcation_y=[config['plot_state']],
        xlabel=config['primary_continuation_parameter'],
        ylabel=config['plot_state'],
        title=f"{experiment['model_name']}: 1D bifurcation plot",
    )


def configure_auto_2d_plot(plotter, experiment, grid=False):
    config = experiment['config']
    plotter.config(
        grid=grid,
        bifurcation_x=[config['secondary_continuation_parameter']],
        bifurcation_y=[config['primary_continuation_parameter']],
        xlabel=config['secondary_continuation_parameter'],
        ylabel=config['primary_continuation_parameter'],
        title='',
        xlim=(float(config['secondary_min']), float(config['secondary_max'])),
        ylim=(
            float(experiment['primary_min']) if config.get('two_d_plot_ymin') is None else float(config['two_d_plot_ymin']),
            float(experiment['primary_max']) if config.get('two_d_plot_ymax') is None else float(config['two_d_plot_ymax']),
        ),
    )


# --- v062 translation verification helpers ---
def build_translation_maps(model):
    param_index = OrderedDict((name, idx) for idx, name in enumerate(model['param_names'], start=1))
    state_index = OrderedDict((name, idx) for idx, name in enumerate(model['state_names'], start=1))
    return param_index, state_index


def render_auto_source_texts(model, config, boundary_state, model_name, primary_min, primary_max):
    param_names = model['param_names']
    state_names = model['state_names']
    param_values = model['param_values']
    alias_defs = model['alias_defs']
    function_defs = model['function_defs']
    equation_defs = model['equation_defs']

    NPAR = max(40, len(param_names))
    param_index, state_index = build_translation_maps(model)
    primary_index = param_index[config['primary_continuation_parameter']]
    secondary_index = param_index[config['secondary_continuation_parameter']]

    decl_lines = []
    for group in [state_names, param_names, [name for name, _ in alias_defs]]:
        if not group:
            continue
        for part in chunked(group, 8):
            decl_lines.append("  DOUBLE PRECISION :: " + ",".join(part))

    state_assignments = [f"  {name}=U({state_index[name]})" for name in state_names]
    param_assignments = [f"  {name}=PAR({param_index[name]})" for name in param_names]
    alias_assignments = [wrap_fortran_assignment(name, expr) for name, expr in alias_defs]
    f_assignments = [wrap_fortran_assignment(f"F({state_index[name]})", equation_defs[name]) for name in state_names]

    internal_functions = []
    for name, args, expr in function_defs:
        lines = [f"  DOUBLE PRECISION FUNCTION {name}({','.join(args)})", "    IMPLICIT NONE"]
        if args:
            for part in chunked(args, 8):
                lines.append("    DOUBLE PRECISION, INTENT(IN) :: " + ",".join(part))
        lines.append(wrap_fortran_assignment(name, expr, indent='    '))
        lines.append(f"  END FUNCTION {name}")
        internal_functions.append('\n'.join(lines))

    stpnt_lines = [f"  PAR(1:{NPAR})=0.D0", "  U(1:NDIM)=0.D0"]
    for name in state_names:
        stpnt_lines.append(f"  U({state_index[name]})={fmt_fortran(boundary_state[state_names.index(name)])}")
    for name in param_names:
        stpnt_lines.append(f"  PAR({param_index[name]})={fmt_fortran(param_values[name])}")
    stpnt_block = '\n'.join(stpnt_lines)

    func_lines = [
        "SUBROUTINE FUNC(NDIM,U,ICP,PAR,IJAC,F,DFDU,DFDP)",
        "  IMPLICIT NONE",
        "  INTEGER, INTENT(IN) :: NDIM, IJAC",
        "  INTEGER, INTENT(IN) :: ICP(*)",
        "  DOUBLE PRECISION, INTENT(IN) :: U(NDIM)",
        "  DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)",
        "  DOUBLE PRECISION, INTENT(OUT) :: F(NDIM)",
        "  DOUBLE PRECISION, INTENT(INOUT) :: DFDU(NDIM,*), DFDP(NDIM,*)",
    ]
    func_lines.extend(decl_lines)
    func_lines.extend(state_assignments)
    func_lines.extend(param_assignments)
    func_lines.extend(alias_assignments)
    func_lines.extend(f_assignments)
    func_lines.append("  RETURN")
    func_lines.append("CONTAINS")
    func_lines.extend(internal_functions)
    func_lines.append("END SUBROUTINE FUNC")
    func_text = '\n'.join(func_lines)

    stpnt_text = dedent(f'''
    SUBROUTINE STPNT(NDIM,U,PAR,T)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM
      DOUBLE PRECISION, INTENT(OUT) :: U(NDIM)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
      DOUBLE PRECISION, INTENT(IN) :: T

    {stpnt_block}
    END SUBROUTINE STPNT
    ''').strip()

    bcnd_text = dedent('''
    SUBROUTINE BCND(NDIM,PAR,ICP,NBC,U0,U1,FB,IJAC,DBC)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM,NBC,IJAC
      INTEGER, INTENT(IN) :: ICP(*)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
      DOUBLE PRECISION, INTENT(IN) :: U0(NDIM),U1(NDIM)
      DOUBLE PRECISION, INTENT(OUT) :: FB(*)
      DOUBLE PRECISION, INTENT(INOUT) :: DBC(*)
    END SUBROUTINE BCND

    SUBROUTINE ICND(NDIM,PAR,ICP,NINT,U,UOLD,UDOT,UPOLD,FI,IJAC,DINT)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM,NINT,IJAC
      INTEGER, INTENT(IN) :: ICP(*)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
      DOUBLE PRECISION, INTENT(IN) :: U(NDIM),UOLD(NDIM),UDOT(NDIM),UPOLD(NDIM)
      DOUBLE PRECISION, INTENT(OUT) :: FI(*)
      DOUBLE PRECISION, INTENT(INOUT) :: DINT(*)
    END SUBROUTINE ICND

    SUBROUTINE FOPT(NDIM,U,ICP,PAR,IJAC,FS,DFDU,DFDP)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM,IJAC
      INTEGER, INTENT(IN) :: ICP(*)
      DOUBLE PRECISION, INTENT(IN) :: U(NDIM)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
      DOUBLE PRECISION, INTENT(OUT) :: FS
      DOUBLE PRECISION, INTENT(INOUT) :: DFDU(*),DFDP(*)
    END SUBROUTINE FOPT

    SUBROUTINE PVLS(NDIM,U,PAR)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM
      DOUBLE PRECISION, INTENT(IN) :: U(NDIM)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
    END SUBROUTINE PVLS
    ''').strip()

    f90_text = func_text + '\n\n' + stpnt_text + '\n\n' + bcnd_text + '\n'

    parnames_text = ', '.join(f"{idx}: '{name}'" for name, idx in param_index.items())
    unames_text = ', '.join(f"{idx}: '{name}'" for name, idx in state_index.items())

    c_text = dedent(f'''
    e = '{model_name}'
    NDIM = {len(state_names)}
    NPAR = {NPAR}
    IPS = 1
    IRS = 0
    ILP = 1
    ICP = [{primary_index}]
    NTST = 20
    NCOL = 4
    IAD = 3
    ISP = 2
    ISW = 1
    IPLT = 0
    NBC = 0
    NINT = 0
    NMX = 1000
    NPR = 50
    MXBF = 10
    IID = 2
    ITMX = 10
    ITNW = 9
    NWTN = 7
    JAC = 0
    EPSL = 1e-8
    EPSU = 1e-8
    EPSS = 1e-6
    DS = 1e-4
    DSMIN = 1e-6
    DSMAX = 2e-2
    IADS = 1
    RL0 = {primary_min}
    RL1 = {primary_max}
    A0 = 0.0
    A1 = 20.0
    UZR = {{}}
    UZSTOP = {{{primary_index}: [{primary_min}, {primary_max}]}}
    SP = ['BP', 'LP', 'HB']
    STOP = []
    parnames = {{{parnames_text}}}
    unames = {{{unames_text}}}
    ''').strip() + "\n"

    return {
        'model_name': model_name,
        'f90_text': f90_text,
        'c_text': c_text,
        'param_index': param_index,
        'state_index': state_index,
        'param_values': param_values,
        'primary_index': primary_index,
        'secondary_index': secondary_index,
        'primary_min': primary_min,
        'primary_max': primary_max,
    }


def verify_auto_source_texts(rendered):
    f90_text = rendered['f90_text']
    c_text = rendered['c_text']
    errors = []

    for name, idx in rendered['param_index'].items():
        pattern = rf'^\s*{re.escape(name)}=PAR\({idx}\)\s*$'
        if re.search(pattern, f90_text, re.MULTILINE) is None:
            errors.append(f"Missing parameter mapping for {name} -> PAR({idx}) in F90.")

        stpnt_pattern = rf'^\s*PAR\({idx}\)={re.escape(fmt_fortran(rendered["param_values"][name]))}\s*$'
        if re.search(stpnt_pattern, f90_text, re.MULTILINE) is None:
            errors.append(f"Missing STPNT assignment for {name} at PAR({idx}).")

        c_pattern = rf"{idx}: '{re.escape(name)}'"
        if re.search(c_pattern, c_text) is None:
            errors.append(f"Missing c-file parnames entry for {name} at index {idx}.")

    for name, idx in rendered['state_index'].items():
        u_pattern = rf'^\s*{re.escape(name)}=U\({idx}\)\s*$'
        if re.search(u_pattern, f90_text, re.MULTILINE) is None:
            errors.append(f"Missing state mapping for {name} -> U({idx}) in F90.")

        f_pattern = rf'^\s*F\({idx}\)\s*='
        if re.search(f_pattern, f90_text, re.MULTILINE) is None:
            errors.append(f"Missing RHS mapping for {name} at F({idx}) in F90.")

        c_pattern = rf"{idx}: '{re.escape(name)}'"
        if re.search(c_pattern, c_text) is None:
            errors.append(f"Missing c-file unames entry for {name} at index {idx}.")

    if re.search(rf'^ICP = \[{rendered["primary_index"]}\]\s*$', c_text, re.MULTILINE) is None:
        errors.append('Primary continuation parameter index is missing from ICP in c-file.')

    uzstop_pattern = re.escape(
        f"UZSTOP = {{{rendered['primary_index']}: [{rendered['primary_min']}, {rendered['primary_max']}]}}"
    )
    if re.search(uzstop_pattern, c_text) is None:
        errors.append('Primary continuation bounds are not tied to the correct parameter index in UZSTOP.')

    return {'ok': not errors, 'errors': errors}


def generate_auto_files(model, config, boundary_state, workdir, model_name, primary_min, primary_max):
    rendered = render_auto_source_texts(model, config, boundary_state, model_name, primary_min, primary_max)

    f90_path = Path(workdir / f"{model_name}.f90")
    c_path = Path(workdir / f"c.{model_name}")
    f90_path.write_text(rendered['f90_text'])
    c_path.write_text(rendered['c_text'])

    verification = verify_auto_source_texts(rendered)
    rendered['verification'] = verification
    rendered['f90_path'] = f90_path
    rendered['c_path'] = c_path

    if not verification['ok']:
        message = '; '.join(verification['errors'][:5])
        raise RuntimeError(f"Generated AUTO files failed translation verification: {message}")

    return rendered


def prepare_experiment(config, ode_text):
    config = normalize_experiment_config(config)
    model_name = config['model_name']
    workdir = Path.cwd() / f"{model_name}_auto"
    workdir.mkdir(parents=True, exist_ok=True)
    os.chdir(workdir)

    Path(f"{model_name}.ode").write_text(ode_text)

    model = parse_model_text(ode_text)
    validate_experiment_config(model, config)

    primary_min = (
        float(model['auto_settings'].get('PARMIN', -0.25))
        if config['primary_min'] is None
        else float(config['primary_min'])
    )
    primary_max = (
        float(model['auto_settings'].get('PARMAX', 0.5))
        if config['primary_max'] is None
        else float(config['primary_max'])
    )

    boundary_state = solve_boundary_equilibrium(model, config, model['param_values'])
    boundary_residual = np.linalg.norm(full_rhs(model, boundary_state, model['param_values']), ord=np.inf)
    linear_thresholds = find_linear_thresholds(
        model,
        config,
        boundary_state,
        model['param_values'],
        primary_min,
        primary_max,
    )

    translation = generate_auto_files(model, config, boundary_state, workdir, model_name, primary_min, primary_max)

    return {
        'config': config,
        'model_name': model_name,
        'workdir': workdir,
        'model': model,
        'primary_min': primary_min,
        'primary_max': primary_max,
        'boundary_state': boundary_state,
        'boundary_residual': boundary_residual,
        'linear_thresholds': linear_thresholds,
        'translation': translation,
        'runner': None,
        'one_d_plot_filename': f"{model_name}_1d_{config['plot_state'].lower()}_vs_{config['primary_continuation_parameter']}.png",
        'two_d_plot_filename': f"{model_name}_2d_{config['secondary_continuation_parameter']}_vs_{config['primary_continuation_parameter']}.png",
    }


def show_translation_summary(experiment, limit=10):
    translation = experiment['translation']
    config = experiment['config']
    verification = translation['verification']

    print(f"Translation verification: {'passed' if verification['ok'] else 'FAILED'}")
    print(
        f"Primary continuation parameter index: "
        f"{config['primary_continuation_parameter']} -> PAR({translation['primary_index']})"
    )
    print(
        f"Secondary continuation parameter index: "
        f"{config['secondary_continuation_parameter']} -> PAR({translation['secondary_index']})"
    )
    print('Parameter order preview:')
    for name, idx in list(translation['param_index'].items())[:limit]:
        print(f"  PAR({idx}) = {name}")
    print('State order:')
    for name, idx in translation['state_index'].items():
        print(f"  U({idx}) = {name}")
    print(f"Generated F90 file: {translation['f90_path']}")
    print(f"Generated c-file: {translation['c_path']}")
    if verification['errors']:
        print('Verification errors:')
        for message in verification['errors']:
            print(f"  - {message}")


def show_experiment_summary(experiment):
    model = experiment['model']
    config = experiment['config']
    print(f"Notebook model name: {experiment['model_name']}")
    print(f"Working directory: {experiment['workdir']}")
    print('Boundary equilibrium:')
    for name, value in zip(model['state_names'], experiment['boundary_state']):
        print(f"  {name}* = {value:.12f}")
    print(f"Boundary residual: {experiment['boundary_residual']:.3e}")
    print('Predicted thresholds:')
    for idx, value in enumerate(experiment['linear_thresholds'], start=1):
        print(f"  root {idx}: {config['primary_continuation_parameter']} = {value:.12f}")
    show_translation_summary(experiment)


# --- v063 report, continuation-limit, and plot cleanup overrides ---
def build_continuation_report_text(experiment):
    config = experiment['config']
    model = experiment['model']
    translation = experiment['translation']

    primary_name = config['primary_continuation_parameter']
    secondary_name = config['secondary_continuation_parameter']
    primary_value = float(model['param_values'][primary_name])

    lines = [
        f"Model name: {experiment['model_name']}",
        "",
        f"Initial value used for continuation: {primary_name} = {primary_value:.16g}",
        "",
        "Parameter order with values assigned that is fed directly into AUTO:",
    ]
    for name, idx in translation['param_index'].items():
        lines.append(f"  PAR({idx}) = {name} = {float(translation['param_values'][name]):.16g}")

    lines.extend(
        [
            "",
            (
                "Primary continuation parameter: "
                f"{primary_name} -> PAR({translation['primary_index']})"
            ),
            (
                "Secondary continuation parameter: "
                f"{secondary_name} -> PAR({translation['secondary_index']})"
            ),
        ]
    )
    return "\n".join(lines) + "\n"


def write_continuation_report(experiment):
    report_path = experiment['workdir'] / f"{experiment['model_name']}_continuation_setup.txt"
    report_path.write_text(build_continuation_report_text(experiment))
    return report_path


def render_auto_source_texts(model, config, boundary_state, model_name, primary_min, primary_max):
    param_names = model['param_names']
    state_names = model['state_names']
    param_values = model['param_values']
    alias_defs = model['alias_defs']
    function_defs = model['function_defs']
    equation_defs = model['equation_defs']

    NPAR = max(40, len(param_names))
    param_index, state_index = build_translation_maps(model)
    primary_index = param_index[config['primary_continuation_parameter']]
    secondary_index = param_index[config['secondary_continuation_parameter']]

    eq_ds = (
        float(model['auto_settings'].get('DS', 1.0e-4))
        if config.get('eq_ds') is None
        else float(config['eq_ds'])
    )
    eq_dsmin = (
        float(model['auto_settings'].get('DSMIN', 1.0e-6))
        if config.get('eq_dsmin') is None
        else float(config['eq_dsmin'])
    )
    eq_dsmax = (
        float(model['auto_settings'].get('DSMAX', 2.0e-2))
        if config.get('eq_dsmax') is None
        else float(config['eq_dsmax'])
    )
    eq_norm_min = float(config.get('eq_norm_min', 0.0))
    eq_norm_max = (
        float(model['auto_settings'].get('NORMMAX', 20.0))
        if config.get('eq_norm_max') is None
        else float(config['eq_norm_max'])
    )

    decl_lines = []
    for group in [state_names, param_names, [name for name, _ in alias_defs]]:
        if not group:
            continue
        for part in chunked(group, 8):
            decl_lines.append("  DOUBLE PRECISION :: " + ",".join(part))

    state_assignments = [f"  {name}=U({state_index[name]})" for name in state_names]
    param_assignments = [f"  {name}=PAR({param_index[name]})" for name in param_names]
    alias_assignments = [wrap_fortran_assignment(name, expr) for name, expr in alias_defs]
    f_assignments = [wrap_fortran_assignment(f"F({state_index[name]})", equation_defs[name]) for name in state_names]

    internal_functions = []
    for name, args, expr in function_defs:
        lines = [f"  DOUBLE PRECISION FUNCTION {name}({','.join(args)})", "    IMPLICIT NONE"]
        if args:
            for part in chunked(args, 8):
                lines.append("    DOUBLE PRECISION, INTENT(IN) :: " + ",".join(part))
        lines.append(wrap_fortran_assignment(name, expr, indent='    '))
        lines.append(f"  END FUNCTION {name}")
        internal_functions.append('\n'.join(lines))

    stpnt_lines = [f"  PAR(1:{NPAR})=0.D0", "  U(1:NDIM)=0.D0"]
    for name in state_names:
        stpnt_lines.append(f"  U({state_index[name]})={fmt_fortran(boundary_state[state_names.index(name)])}")
    for name in param_names:
        stpnt_lines.append(f"  PAR({param_index[name]})={fmt_fortran(param_values[name])}")
    stpnt_block = '\n'.join(stpnt_lines)

    func_lines = [
        "SUBROUTINE FUNC(NDIM,U,ICP,PAR,IJAC,F,DFDU,DFDP)",
        "  IMPLICIT NONE",
        "  INTEGER, INTENT(IN) :: NDIM, IJAC",
        "  INTEGER, INTENT(IN) :: ICP(*)",
        "  DOUBLE PRECISION, INTENT(IN) :: U(NDIM)",
        "  DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)",
        "  DOUBLE PRECISION, INTENT(OUT) :: F(NDIM)",
        "  DOUBLE PRECISION, INTENT(INOUT) :: DFDU(NDIM,*), DFDP(NDIM,*)",
    ]
    func_lines.extend(decl_lines)
    func_lines.extend(state_assignments)
    func_lines.extend(param_assignments)
    func_lines.extend(alias_assignments)
    func_lines.extend(f_assignments)
    func_lines.append("  RETURN")
    func_lines.append("CONTAINS")
    func_lines.extend(internal_functions)
    func_lines.append("END SUBROUTINE FUNC")
    func_text = '\n'.join(func_lines)

    stpnt_text = dedent(f'''
    SUBROUTINE STPNT(NDIM,U,PAR,T)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM
      DOUBLE PRECISION, INTENT(OUT) :: U(NDIM)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
      DOUBLE PRECISION, INTENT(IN) :: T

    {stpnt_block}
    END SUBROUTINE STPNT
    ''').strip()

    bcnd_text = dedent('''
    SUBROUTINE BCND(NDIM,PAR,ICP,NBC,U0,U1,FB,IJAC,DBC)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM,NBC,IJAC
      INTEGER, INTENT(IN) :: ICP(*)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
      DOUBLE PRECISION, INTENT(IN) :: U0(NDIM),U1(NDIM)
      DOUBLE PRECISION, INTENT(OUT) :: FB(*)
      DOUBLE PRECISION, INTENT(INOUT) :: DBC(*)
    END SUBROUTINE BCND

    SUBROUTINE ICND(NDIM,PAR,ICP,NINT,U,UOLD,UDOT,UPOLD,FI,IJAC,DINT)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM,NINT,IJAC
      INTEGER, INTENT(IN) :: ICP(*)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
      DOUBLE PRECISION, INTENT(IN) :: U(NDIM),UOLD(NDIM),UDOT(NDIM),UPOLD(NDIM)
      DOUBLE PRECISION, INTENT(OUT) :: FI(*)
      DOUBLE PRECISION, INTENT(INOUT) :: DINT(*)
    END SUBROUTINE ICND

    SUBROUTINE FOPT(NDIM,U,ICP,PAR,IJAC,FS,DFDU,DFDP)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM,IJAC
      INTEGER, INTENT(IN) :: ICP(*)
      DOUBLE PRECISION, INTENT(IN) :: U(NDIM)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
      DOUBLE PRECISION, INTENT(OUT) :: FS
      DOUBLE PRECISION, INTENT(INOUT) :: DFDU(*),DFDP(*)
    END SUBROUTINE FOPT

    SUBROUTINE PVLS(NDIM,U,PAR)
      IMPLICIT NONE
      INTEGER, INTENT(IN) :: NDIM
      DOUBLE PRECISION, INTENT(IN) :: U(NDIM)
      DOUBLE PRECISION, INTENT(INOUT) :: PAR(*)
    END SUBROUTINE PVLS
    ''').strip()

    f90_text = func_text + '\n\n' + stpnt_text + '\n\n' + bcnd_text + '\n'

    parnames_text = ', '.join(f"{idx}: '{name}'" for name, idx in param_index.items())
    unames_text = ', '.join(f"{idx}: '{name}'" for name, idx in state_index.items())

    c_text = dedent(f'''
    e = '{model_name}'
    NDIM = {len(state_names)}
    NPAR = {NPAR}
    IPS = 1
    IRS = 0
    ILP = 1
    ICP = [{primary_index}]
    NTST = 20
    NCOL = 4
    IAD = 3
    ISP = 2
    ISW = 1
    IPLT = 0
    NBC = 0
    NINT = 0
    NMX = 1000
    NPR = 50
    MXBF = 10
    IID = 2
    ITMX = 10
    ITNW = 9
    NWTN = 7
    JAC = 0
    EPSL = 1e-8
    EPSU = 1e-8
    EPSS = 1e-6
    DS = {eq_ds}
    DSMIN = {eq_dsmin}
    DSMAX = {eq_dsmax}
    IADS = 1
    RL0 = {primary_min}
    RL1 = {primary_max}
    A0 = {eq_norm_min}
    A1 = {eq_norm_max}
    UZR = {{}}
    UZSTOP = {{{primary_index}: [{primary_min}, {primary_max}]}}
    SP = ['BP', 'LP', 'HB']
    STOP = []
    parnames = {{{parnames_text}}}
    unames = {{{unames_text}}}
    ''').strip() + "\n"

    return {
        'model_name': model_name,
        'f90_text': f90_text,
        'c_text': c_text,
        'param_index': param_index,
        'state_index': state_index,
        'param_values': param_values,
        'primary_index': primary_index,
        'secondary_index': secondary_index,
        'primary_min': primary_min,
        'primary_max': primary_max,
        'eq_ds': eq_ds,
        'eq_dsmin': eq_dsmin,
        'eq_dsmax': eq_dsmax,
        'eq_norm_min': eq_norm_min,
        'eq_norm_max': eq_norm_max,
    }


class HeadlessAutoPlotter:
    def __init__(self, name=None, templates=None, **kw):
        self.diagram = ac.loadbd(name) if isinstance(name, str) else name
        self.options = {
            'stability': False,
            'grid': False,
            'bifurcation_x': [0],
            'bifurcation_y': [1],
            'xlabel': '',
            'ylabel': '',
            'title': '',
            'minx': None,
            'maxx': None,
            'miny': None,
            'maxy': None,
        }
        if kw:
            self.config(**kw)


    def config(self, **kw):
        self.options.update(kw)
        return self


    def _branches(self):
        return list(getattr(self.diagram, 'branches', self.diagram))


    def _coordnames(self):
        for branch in self._branches():
            coordnames = getattr(branch, 'coordnames', None)
            if coordnames:
                return list(coordnames)
        return []


    def _resolve_columns(self, spec):
        coordnames = self._coordnames()
        columns = spec
        if columns is None:
            return []
        if not isinstance(columns, (list, tuple)):
            columns = [columns]
        resolved = []
        for item in columns:
            if isinstance(item, int):
                resolved.append(item)
            else:
                resolved.append(coordnames.index(str(item)))
        return resolved


    def savefig(self, filename):
        plt = get_matplotlib_pyplot()
        xcolumns = self._resolve_columns(self.options.get('bifurcation_x'))
        ycolumns = self._resolve_columns(self.options.get('bifurcation_y'))
        branches = self._branches()
        fig, ax = plt.subplots(figsize=(6.4, 4.8))
        colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', ['C0'])

        for branch_index, branch in enumerate(branches):
            color = colors[branch_index % len(colors)]
            for xcolumn, ycolumn in zip(xcolumns, ycolumns):
                x = np.asarray(branch.coordarray[xcolumn], dtype=float)
                y = np.asarray(branch.coordarray[ycolumn], dtype=float)

                if self.options.get('stability', False):
                    old = 0
                    stability = branch.stability()
                    if not stability:
                        stability = [len(x)]
                    for point in stability:
                        abs_point = abs(int(point))
                        if abs_point > 1 or point == stability[-1]:
                            xs = x[old:abs_point]
                            ys = y[old:abs_point]
                            if len(xs) >= 2:
                                ax.plot(
                                    xs,
                                    ys,
                                    color=color,
                                    linestyle='-' if point < 0 else '--',
                                    linewidth=1.6,
                                )
                            old = max(abs_point - 1, 0)
                else:
                    if len(x) >= 2:
                        ax.plot(x, y, color=color, linewidth=1.6)

        if self.options.get('minx') is not None and self.options.get('maxx') is not None:
            ax.set_xlim(float(self.options['minx']), float(self.options['maxx']))
        if self.options.get('miny') is not None and self.options.get('maxy') is not None:
            ax.set_ylim(float(self.options['miny']), float(self.options['maxy']))

        ax.set_xlabel(self.options.get('xlabel', ''))
        ax.set_ylabel(self.options.get('ylabel', ''))
        ax.set_title(self.options.get('title', ''))
        ax.grid(bool(self.options.get('grid', False)))
        fig.tight_layout()
        fig.savefig(filename, dpi=150, bbox_inches='tight')
        plt.close(fig)


def prepare_experiment(config, ode_text):
    config = normalize_experiment_config(config)
    model_name = config['model_name']
    workdir = Path.cwd() / f"{model_name}_auto"
    workdir.mkdir(parents=True, exist_ok=True)
    os.chdir(workdir)

    Path(f"{model_name}.ode").write_text(ode_text)

    model = parse_model_text(ode_text)
    validate_experiment_config(model, config)

    primary_min = (
        float(model['auto_settings'].get('PARMIN', -0.25))
        if config['primary_min'] is None
        else float(config['primary_min'])
    )
    primary_max = (
        float(model['auto_settings'].get('PARMAX', 0.5))
        if config['primary_max'] is None
        else float(config['primary_max'])
    )

    boundary_state = solve_boundary_equilibrium(model, config, model['param_values'])
    boundary_residual = np.linalg.norm(full_rhs(model, boundary_state, model['param_values']), ord=np.inf)
    linear_thresholds = find_linear_thresholds(
        model,
        config,
        boundary_state,
        model['param_values'],
        primary_min,
        primary_max,
    )

    translation = generate_auto_files(model, config, boundary_state, workdir, model_name, primary_min, primary_max)

    experiment = {
        'config': config,
        'model_name': model_name,
        'workdir': workdir,
        'model': model,
        'primary_min': primary_min,
        'primary_max': primary_max,
        'boundary_state': boundary_state,
        'boundary_residual': boundary_residual,
        'linear_thresholds': linear_thresholds,
        'translation': translation,
        'runner': None,
        'one_d_plot_filename': f"{model_name}_1d_{config['plot_state'].lower()}_vs_{config['primary_continuation_parameter']}.png",
        'two_d_plot_filename': f"{model_name}_2d_{config['secondary_continuation_parameter']}_vs_{config['primary_continuation_parameter']}.png",
    }
    experiment['continuation_report_path'] = write_continuation_report(experiment)
    return experiment


def configure_auto_1d_plot(plotter, experiment, stability=True, grid=False):
    config = experiment['config']
    x_min = experiment['primary_min'] if config.get('one_d_plot_min') is None else float(config['one_d_plot_min'])
    x_max = experiment['primary_max'] if config.get('one_d_plot_max') is None else float(config['one_d_plot_max'])
    plotter.config(
        stability=stability,
        grid=grid,
        bifurcation_x=[config['primary_continuation_parameter']],
        bifurcation_y=[config['plot_state']],
        xlabel=config['primary_continuation_parameter'],
        ylabel=config['plot_state'],
        title=f"{experiment['model_name']}: 1D bifurcation plot",
        use_labels=False,
        use_symbols=False,
        minx=x_min,
        maxx=x_max,
    )


def configure_auto_2d_plot(plotter, experiment, grid=False):
    config = experiment['config']
    plotter.config(
        grid=grid,
        bifurcation_x=[config['secondary_continuation_parameter']],
        bifurcation_y=[config['primary_continuation_parameter']],
        xlabel=config['secondary_continuation_parameter'],
        ylabel=config['primary_continuation_parameter'],
        title=f"{experiment['model_name']}: 2D bifurcation plot",
        use_labels=False,
        use_symbols=False,
        minx=float(config['secondary_min']),
        maxx=float(config['secondary_max']),
        miny=float(experiment['primary_min']),
        maxy=float(experiment['primary_max']),
    )


# --- v064 ecological 1D plotting overrides ---
def iter_branch_segments(branch, x):
    stability = branch.stability()
    if not stability:
        stability = [len(x)]

    old = 0
    for point in stability:
        abs_point = abs(int(point))
        if abs_point > 1 or point == stability[-1]:
            start = old
            stop = abs_point
            if stop - start >= 2:
                yield start, stop, (point < 0)
            old = max(abs_point - 1, 0)


def classify_ecological_segment(branch, start, stop, config, tol):
    coordnames = list(getattr(branch, 'coordnames', []))

    def values(name):
        idx = coordnames.index(name)
        return np.asarray(branch.coordarray[idx][start:stop], dtype=float)

    positive_vars = [name for name in config.get('positive_branch_variables', []) if name in coordnames]
    zero_vars = [name for name in config.get('zero_branch_variables', []) if name in coordnames]

    if positive_vars:
        is_positive = all(np.max(values(name)) > tol and np.min(values(name)) >= -tol for name in positive_vars)
        if is_positive:
            return 'positive'

    if zero_vars:
        is_zero = all(np.max(np.abs(values(name))) <= tol for name in zero_vars)
        if is_zero:
            return 'zero'

    plot_state_values = values(config['plot_state']) if config['plot_state'] in coordnames else None
    if plot_state_values is not None and np.max(plot_state_values) <= tol:
        return 'nonpositive'
    return 'other'


def save_ecological_1d_plot(experiment, filename=None):
    plt = get_matplotlib_pyplot()
    config = experiment['config']
    diagram = experiment['eq_diagram']
    branches = list(getattr(diagram, 'branches', diagram))
    tol = float(config.get('ecology_plot_tol', 1.0e-8))

    x_name = config['primary_continuation_parameter']
    y_name = config['plot_state']
    output_path = (
        experiment['workdir'] / experiment['one_d_plot_filename']
        if filename is None
        else Path(filename)
    )

    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    drew_anything = False

    for branch in branches:
        coordnames = list(getattr(branch, 'coordnames', []))
        if x_name not in coordnames or y_name not in coordnames:
            continue

        x_idx = coordnames.index(x_name)
        y_idx = coordnames.index(y_name)
        x = np.asarray(branch.coordarray[x_idx], dtype=float)
        y = np.asarray(branch.coordarray[y_idx], dtype=float)

        for start, stop, stable in iter_branch_segments(branch, x):
            xs = x[start:stop]
            ys = y[start:stop]
            segment_type = classify_ecological_segment(branch, start, stop, config, tol)

            if segment_type == 'positive':
                linestyle = '-' if config.get('positive_branch_force_solid', True) else ('-' if stable else '--')
                color = config.get('positive_branch_color', 'red')
                linewidth = float(config.get('positive_branch_linewidth', 1.8))
            elif segment_type == 'zero':
                if not stable and not config.get('show_zero_branch_unstable', False):
                    continue
                if not config.get('show_zero_branch', True):
                    continue
                linestyle = '-' if stable else '--'
                color = config.get('zero_branch_color', 'black')
                linewidth = float(config.get('zero_branch_linewidth', 1.6))
            else:
                if not config.get('show_other_branches', False):
                    continue
                linestyle = '-' if stable else '--'
                color = config.get('other_branch_color', '0.55')
                linewidth = float(config.get('other_branch_linewidth', 1.3))

            ax.plot(xs, ys, color=color, linestyle=linestyle, linewidth=linewidth)
            drew_anything = True

    if not drew_anything:
        raise RuntimeError('Ecological 1D plotter did not find any visible segments to draw.')

    x_min = experiment['primary_min'] if config.get('one_d_plot_min') is None else float(config['one_d_plot_min'])
    x_max = experiment['primary_max'] if config.get('one_d_plot_max') is None else float(config['one_d_plot_max'])
    ax.set_xlim(float(x_min), float(x_max))
    ax.set_xlabel(x_name)
    ax.set_ylabel(y_name)
    ax.set_title(f"{experiment['model_name']}: 1D bifurcation plot")
    ax.grid(bool(config.get('one_d_plot_grid', False)))
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return output_path


# --- v065 backward-S bistability plotting overrides ---
def branch_to_named_arrays(branch):
    coordnames = list(getattr(branch, 'coordnames', []))
    return {
        name: np.asarray(branch.coordarray[idx], dtype=float)
        for idx, name in enumerate(coordnames)
    }


def stability_along_branch(model, primary_name, primary_values, state_matrix, p, tol=1.0e-7):
    stable_mask = []
    max_real_parts = []

    for primary_value, state in zip(primary_values, state_matrix):
        params = dict(p)
        params[primary_name] = float(primary_value)
        eigvals = np.linalg.eigvals(full_jacobian(model, np.asarray(state, dtype=float), params))
        max_real = float(np.max(eigvals.real))
        max_real_parts.append(max_real)
        stable_mask.append(max_real <= tol)

    return np.asarray(stable_mask, dtype=bool), np.asarray(max_real_parts, dtype=float)


def contiguous_segments(mask):
    if len(mask) == 0:
        return []
    segments = []
    start = 0
    for idx in range(1, len(mask)):
        if bool(mask[idx]) != bool(mask[idx - 1]):
            segments.append((start, idx))
            start = idx - 1
    segments.append((start, len(mask)))
    return segments


def extract_bistability_branches(experiment):
    config = experiment['config']
    model = experiment['model']
    primary_name = config['primary_continuation_parameter']
    zero_tol = float(config.get('branch_zero_tol', 1.0e-7))
    nonneg_tol = float(config.get('branch_nonnegative_tol', 1.0e-7))
    min_points = int(config.get('branch_min_points', 5))

    candidate_bottom = []
    candidate_positive = []
    branches = list(getattr(experiment['eq_diagram'], 'branches', experiment['eq_diagram']))

    for idx, branch in enumerate(branches, start=1):
        arrays = branch_to_named_arrays(branch)
        if primary_name not in arrays:
            continue
        if any(name not in arrays for name in model['state_names']):
            continue

        primary_values = np.asarray(arrays[primary_name], dtype=float)
        if len(primary_values) < min_points:
            continue

        state_matrix = np.column_stack([np.asarray(arrays[name], dtype=float) for name in model['state_names']])

        if all(np.max(np.abs(np.asarray(arrays[name], dtype=float))) < zero_tol for name in config['zero_branch_variables']):
            candidate_bottom.append(
                {
                    'index': idx,
                    'primary': primary_values,
                    'states': state_matrix,
                    'plot_values': np.asarray(arrays[config['plot_state']], dtype=float),
                }
            )
            continue

        if (
            all(np.min(np.asarray(arrays[name], dtype=float)) >= -nonneg_tol for name in config['positive_branch_variables'])
            and max(np.max(np.asarray(arrays[name], dtype=float)) for name in config['positive_branch_variables']) > zero_tol
        ):
            candidate_positive.append(
                {
                    'index': idx,
                    'primary': primary_values,
                    'states': state_matrix,
                    'plot_values': np.asarray(arrays[config['plot_state']], dtype=float),
                }
            )

    if not candidate_positive:
        raise RuntimeError('Could not find a nonnegative positive-equilibrium branch for the 1D bistability plot.')

    bottom_branch = max(candidate_bottom, key=lambda item: len(item['primary'])) if candidate_bottom else None
    positive_branch = max(
        candidate_positive,
        key=lambda item: (len(item['primary']), float(np.max(item['plot_values']))),
    )
    return bottom_branch, positive_branch


def newton_equilibrium_at_primary(model, primary_name, primary_value, state_guess, p, max_it=30, tol=1.0e-12):
    x = np.asarray(state_guess, dtype=float).copy()

    for _ in range(max_it):
        params = dict(p)
        params[primary_name] = float(primary_value)
        residual = full_rhs(model, x, params)
        jac = full_jacobian(model, x, params)
        step = np.linalg.solve(jac, -residual)
        x = x + step

        if np.linalg.norm(step, ord=np.inf) < tol and np.linalg.norm(residual, ord=np.inf) < 1.0e-9:
            return x, True, float(np.max(np.abs(residual)))

    return x, False, float(np.max(np.abs(residual)))


def build_upper_branch_rescue(experiment, positive_branch):
    config = experiment['config']
    model = experiment['model']
    primary_name = config['primary_continuation_parameter']
    plot_index = model['state_names'].index(config['plot_state'])
    positive_indices = [model['state_names'].index(name) for name in config['positive_branch_variables']]

    step = float(config.get('rescue_step', 0.005))
    primary_stop = float(config.get('rescue_primary_stop', 0.01))
    positive_tol = float(config.get('rescue_positive_tol', 1.0e-8))
    monotonic_tol = float(config.get('rescue_monotonic_tol', 1.0e-8))
    max_it = int(config.get('rescue_max_it', 30))
    newton_tol = float(config.get('rescue_newton_tol', 1.0e-12))

    endpoint_primary = float(positive_branch['primary'][-1])
    endpoint_state = np.asarray(positive_branch['states'][-1], dtype=float).copy()

    targets = np.arange(endpoint_primary - step, primary_stop - 0.5 * step, -step)
    rescue_primary = []
    rescue_states = []
    rescue_residuals = []

    current_state = endpoint_state
    current_plot = float(endpoint_state[plot_index])

    for primary_target in targets:
        candidate_state, converged, max_residual = newton_equilibrium_at_primary(
            model,
            primary_name,
            primary_target,
            current_state,
            model['param_values'],
            max_it=max_it,
            tol=newton_tol,
        )
        if not converged:
            break

        if any(float(candidate_state[idx]) <= positive_tol for idx in positive_indices):
            break

        candidate_plot = float(candidate_state[plot_index])
        if candidate_plot <= current_plot + monotonic_tol:
            break

        rescue_primary.append(float(primary_target))
        rescue_states.append(candidate_state.copy())
        rescue_residuals.append(float(max_residual))
        current_state = candidate_state
        current_plot = candidate_plot

    if not rescue_primary:
        empty = np.empty((0,), dtype=float)
        return {
            'primary': empty,
            'states': np.empty((0, len(model['state_names'])), dtype=float),
            'plot_values': empty,
            'max_residual': empty,
        }

    rescue_states = np.asarray(rescue_states, dtype=float)
    return {
        'primary': np.asarray(rescue_primary, dtype=float),
        'states': rescue_states,
        'plot_values': rescue_states[:, plot_index],
        'max_residual': np.asarray(rescue_residuals, dtype=float),
    }


def save_backward_s_1d_plot(experiment, filename=None):
    plt = get_matplotlib_pyplot()
    config = experiment['config']
    model = experiment['model']
    primary_name = config['primary_continuation_parameter']
    plot_state = config['plot_state']

    _, positive_branch = extract_bistability_branches(experiment)
    upper_rescue = build_upper_branch_rescue(experiment, positive_branch)

    positive_stable_mask, _ = stability_along_branch(
        model,
        primary_name,
        positive_branch['primary'],
        positive_branch['states'],
        model['param_values'],
        tol=float(config.get('stability_tol', 1.0e-7)),
    )

    if len(upper_rescue['primary']) > 0:
        rescue_stable_mask, _ = stability_along_branch(
            model,
            primary_name,
            upper_rescue['primary'],
            upper_rescue['states'],
            model['param_values'],
            tol=float(config.get('stability_tol', 1.0e-7)),
        )
    else:
        rescue_stable_mask = np.empty((0,), dtype=bool)

    output_path = (
        experiment['workdir'] / experiment['one_d_plot_filename']
        if filename is None
        else Path(filename)
    )

    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    stable_color = config.get('stable_branch_color', 'tab:blue')
    unstable_color = config.get('unstable_branch_color', 'tab:red')
    stable_lw = float(config.get('stable_branch_linewidth', 2.6))
    unstable_lw = float(config.get('unstable_branch_linewidth', 2.1))

    for start, stop in contiguous_segments(positive_stable_mask):
        stable = bool(positive_stable_mask[start])
        ax.plot(
            positive_branch['primary'][start:stop],
            positive_branch['plot_values'][start:stop],
            color=stable_color if stable else unstable_color,
            lw=stable_lw if stable else unstable_lw,
            ls='-' if stable else '--',
        )

    if len(upper_rescue['primary']) > 0:
        rescue_order = np.argsort(upper_rescue['primary'])
        rescue_primary = upper_rescue['primary'][rescue_order]
        rescue_plot = upper_rescue['plot_values'][rescue_order]
        rescue_mask = rescue_stable_mask[rescue_order]

        for start, stop in contiguous_segments(rescue_mask):
            if not bool(rescue_mask[start]):
                continue
            ax.plot(
                rescue_primary[start:stop],
                rescue_plot[start:stop],
                color=stable_color,
                lw=stable_lw,
                ls='-',
            )

    x_min = float(config.get('bistability_plot_min', 0.0))
    x_max = experiment['primary_max'] if config.get('bistability_plot_max') is None else float(config['bistability_plot_max'])

    ymax_candidates = [float(np.max(positive_branch['plot_values']))]
    if len(upper_rescue['plot_values']) > 0:
        ymax_candidates.append(float(np.max(upper_rescue['plot_values'])))

    ax.set_xlabel(primary_name)
    ax.set_ylabel(plot_state)
    ax.set_xlim(x_min, float(x_max))
    ax.set_ylim(float(config.get('bistability_plot_ymin', -0.02)), 1.05 * max(ymax_candidates))
    ax.grid(bool(config.get('bistability_plot_grid', False)))
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return output_path


# --- v066 plain-style and headless-2D overrides ---
def patched_auto_plot(name=None, templates=None, **kw):
    plotter = _AUTO_PLOT(name, templates, **kw)
    if plotter is not None and getattr(plotter, 'tk', None) is not None:
        return plotter
    return HeadlessAutoPlotter(name, templates=templates, **kw)


ac.plot = patched_auto_plot


def save_backward_s_1d_plot(experiment, filename=None):
    plt = get_matplotlib_pyplot()
    config = experiment['config']
    model = experiment['model']
    primary_name = config['primary_continuation_parameter']
    plot_state = config['plot_state']

    _, positive_branch = extract_bistability_branches(experiment)
    upper_rescue = build_upper_branch_rescue(experiment, positive_branch)

    positive_stable_mask, _ = stability_along_branch(
        model,
        primary_name,
        positive_branch['primary'],
        positive_branch['states'],
        model['param_values'],
        tol=float(config.get('stability_tol', 1.0e-7)),
    )

    if len(upper_rescue['primary']) > 0:
        rescue_stable_mask, _ = stability_along_branch(
            model,
            primary_name,
            upper_rescue['primary'],
            upper_rescue['states'],
            model['param_values'],
            tol=float(config.get('stability_tol', 1.0e-7)),
        )
    else:
        rescue_stable_mask = np.empty((0,), dtype=bool)

    output_path = (
        experiment['workdir'] / experiment['one_d_plot_filename']
        if filename is None
        else Path(filename)
    )

    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    stable_color = config.get('stable_branch_color', 'black')
    unstable_color = config.get('unstable_branch_color', 'black')
    stable_lw = float(config.get('stable_branch_linewidth', 2.6))
    unstable_lw = float(config.get('unstable_branch_linewidth', 2.1))

    for start, stop in contiguous_segments(positive_stable_mask):
        stable = bool(positive_stable_mask[start])
        ax.plot(
            positive_branch['primary'][start:stop],
            positive_branch['plot_values'][start:stop],
            color=stable_color if stable else unstable_color,
            lw=stable_lw if stable else unstable_lw,
            ls='-' if stable else '--',
        )

    if len(upper_rescue['primary']) > 0:
        rescue_order = np.argsort(upper_rescue['primary'])
        rescue_primary = upper_rescue['primary'][rescue_order]
        rescue_plot = upper_rescue['plot_values'][rescue_order]
        rescue_mask = rescue_stable_mask[rescue_order]

        for start, stop in contiguous_segments(rescue_mask):
            if not bool(rescue_mask[start]):
                continue
            ax.plot(
                rescue_primary[start:stop],
                rescue_plot[start:stop],
                color=stable_color,
                lw=stable_lw,
                ls='-',
            )

    x_min = float(config.get('bistability_plot_min', 0.0))
    x_max = experiment['primary_max'] if config.get('bistability_plot_max') is None else float(config['bistability_plot_max'])

    ymax_candidates = [float(np.max(positive_branch['plot_values']))]
    if len(upper_rescue['plot_values']) > 0:
        ymax_candidates.append(float(np.max(upper_rescue['plot_values'])))

    ax.set_xlabel(primary_name)
    ax.set_ylabel(plot_state)
    ax.set_xlim(x_min, float(x_max))
    ax.set_ylim(float(config.get('bistability_plot_ymin', -0.02)), 1.05 * max(ymax_candidates))
    ax.grid(bool(config.get('bistability_plot_grid', False)))
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return output_path


# --- v067 backward-S assembly overrides ---
def _nonoverlapping_segments(mask):
    mask = np.asarray(mask, dtype=bool)
    if len(mask) == 0:
        return []
    segments = []
    start = 0
    current = bool(mask[0])
    for idx in range(1, len(mask)):
        if bool(mask[idx]) != current:
            segments.append((start, idx, current))
            start = idx
            current = bool(mask[idx])
    segments.append((start, len(mask), current))
    return segments


def save_backward_s_1d_plot(experiment, filename=None):
    plt = get_matplotlib_pyplot()
    config = experiment['config']
    model = experiment['model']
    primary_name = config['primary_continuation_parameter']
    plot_state = config['plot_state']

    _, positive_branch = extract_bistability_branches(experiment)
    upper_rescue = build_upper_branch_rescue(experiment, positive_branch)

    positive_stable_mask, _ = stability_along_branch(
        model,
        primary_name,
        positive_branch['primary'],
        positive_branch['states'],
        model['param_values'],
        tol=float(config.get('stability_tol', 1.0e-7)),
    )

    positive_primary = np.asarray(positive_branch['primary'], dtype=float)
    positive_plot = np.asarray(positive_branch['plot_values'], dtype=float)

    unstable_indices = np.flatnonzero(~positive_stable_mask)
    lower_stop = len(positive_primary)
    middle_start = None
    middle_stop = None

    if len(unstable_indices) > 0:
        middle_start = int(unstable_indices[0])
        middle_stop = int(unstable_indices[-1]) + 1
        lower_stop = max(middle_start + 1, 2)

    lower_primary = positive_primary[:lower_stop]
    lower_plot = positive_plot[:lower_stop]

    output_path = (
        experiment['workdir'] / experiment['one_d_plot_filename']
        if filename is None
        else Path(filename)
    )

    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    stable_color = config.get('stable_branch_color', 'black')
    unstable_color = config.get('unstable_branch_color', 'black')
    stable_lw = float(config.get('stable_branch_linewidth', 2.6))
    unstable_lw = float(config.get('unstable_branch_linewidth', 2.1))

    if len(lower_primary) >= 2:
        ax.plot(
            lower_primary,
            lower_plot,
            color=stable_color,
            lw=stable_lw,
            ls='-',
        )

    if middle_start is not None and middle_stop is not None and (middle_stop - middle_start) >= 2:
        ax.plot(
            positive_primary[middle_start:middle_stop],
            positive_plot[middle_start:middle_stop],
            color=unstable_color,
            lw=unstable_lw,
            ls='--',
        )

    tail_start = middle_stop if middle_stop is not None else 0
    if tail_start < len(positive_primary):
        tail_mask = positive_stable_mask[tail_start:]
        for start, stop, is_stable in _nonoverlapping_segments(tail_mask):
            if not is_stable or (stop - start) < 2:
                continue
            ax.plot(
                positive_primary[tail_start + start:tail_start + stop],
                positive_plot[tail_start + start:tail_start + stop],
                color=stable_color,
                lw=stable_lw,
                ls='-',
            )

    if len(upper_rescue['primary']) > 0:
        rescue_stable_mask, _ = stability_along_branch(
            model,
            primary_name,
            upper_rescue['primary'],
            upper_rescue['states'],
            model['param_values'],
            tol=float(config.get('stability_tol', 1.0e-7)),
        )
        rescue_order = np.argsort(upper_rescue['primary'])
        rescue_primary = upper_rescue['primary'][rescue_order]
        rescue_plot = upper_rescue['plot_values'][rescue_order]
        rescue_mask = rescue_stable_mask[rescue_order]

        for start, stop, is_stable in _nonoverlapping_segments(rescue_mask):
            if not is_stable or (stop - start) < 2:
                continue
            ax.plot(
                rescue_primary[start:stop],
                rescue_plot[start:stop],
                color=stable_color,
                lw=stable_lw,
                ls='-',
            )

    x_min = float(config.get('bistability_plot_min', 0.0))
    x_max = (
        experiment['primary_max']
        if config.get('bistability_plot_max') is None
        else float(config['bistability_plot_max'])
    )

    ymax_candidates = [float(np.max(positive_plot))]
    if len(upper_rescue['plot_values']) > 0:
        ymax_candidates.append(float(np.max(upper_rescue['plot_values'])))

    ax.set_xlabel(primary_name)
    ax.set_ylabel(plot_state)
    ax.set_xlim(x_min, float(x_max))
    ax.set_ylim(float(config.get('bistability_plot_ymin', -0.02)), 1.05 * max(ymax_candidates))
    ax.grid(bool(config.get('bistability_plot_grid', False)))
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return output_path


# --- v068 three-role backward-S plotting override ---
def save_backward_s_1d_plot(experiment, filename=None):
    plt = get_matplotlib_pyplot()
    config = experiment['config']
    model = experiment['model']
    primary_name = config['primary_continuation_parameter']
    plot_state = config['plot_state']

    bottom_branch, positive_branch = extract_bistability_branches(experiment)
    upper_rescue = build_upper_branch_rescue(experiment, positive_branch)

    positive_stable_mask, _ = stability_along_branch(
        model,
        primary_name,
        positive_branch['primary'],
        positive_branch['states'],
        model['param_values'],
        tol=float(config.get('stability_tol', 1.0e-7)),
    )

    if bottom_branch is not None:
        bottom_stable_mask, _ = stability_along_branch(
            model,
            primary_name,
            bottom_branch['primary'],
            bottom_branch['states'],
            model['param_values'],
            tol=float(config.get('stability_tol', 1.0e-7)),
        )
    else:
        bottom_stable_mask = np.empty((0,), dtype=bool)

    output_path = (
        experiment['workdir'] / experiment['one_d_plot_filename']
        if filename is None
        else Path(filename)
    )

    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    stable_color = config.get('stable_branch_color', 'black')
    unstable_color = config.get('unstable_branch_color', 'black')
    stable_lw = float(config.get('stable_branch_linewidth', 2.6))
    unstable_lw = float(config.get('unstable_branch_linewidth', 2.1))

    if bottom_branch is not None:
        bottom_primary = np.asarray(bottom_branch['primary'], dtype=float)
        bottom_plot = np.asarray(bottom_branch['plot_values'], dtype=float)
        for start, stop, is_stable in _nonoverlapping_segments(bottom_stable_mask):
            if not is_stable or (stop - start) < 2:
                continue
            ax.plot(
                bottom_primary[start:stop],
                bottom_plot[start:stop],
                color=stable_color,
                lw=stable_lw,
                ls='-',
            )

    positive_primary = np.asarray(positive_branch['primary'], dtype=float)
    positive_plot = np.asarray(positive_branch['plot_values'], dtype=float)
    for start, stop, is_stable in _nonoverlapping_segments(positive_stable_mask):
        if (stop - start) < 2:
            continue
        if is_stable:
            ax.plot(
                positive_primary[start:stop],
                positive_plot[start:stop],
                color=stable_color,
                lw=stable_lw,
                ls='-',
            )
        else:
            ax.plot(
                positive_primary[start:stop],
                positive_plot[start:stop],
                color=unstable_color,
                lw=unstable_lw,
                ls='--',
            )

    if len(upper_rescue['primary']) > 0:
        rescue_stable_mask, _ = stability_along_branch(
            model,
            primary_name,
            upper_rescue['primary'],
            upper_rescue['states'],
            model['param_values'],
            tol=float(config.get('stability_tol', 1.0e-7)),
        )
        rescue_order = np.argsort(upper_rescue['primary'])
        rescue_primary = upper_rescue['primary'][rescue_order]
        rescue_plot = upper_rescue['plot_values'][rescue_order]
        rescue_mask = rescue_stable_mask[rescue_order]

        for start, stop, is_stable in _nonoverlapping_segments(rescue_mask):
            if not is_stable or (stop - start) < 2:
                continue
            ax.plot(
                rescue_primary[start:stop],
                rescue_plot[start:stop],
                color=stable_color,
                lw=stable_lw,
                ls='-',
            )

    x_min = float(config.get('bistability_plot_min', 0.0))
    x_max = (
        experiment['primary_max']
        if config.get('bistability_plot_max') is None
        else float(config['bistability_plot_max'])
    )

    ymax_candidates = [float(np.max(positive_plot))]
    if len(upper_rescue['plot_values']) > 0:
        ymax_candidates.append(float(np.max(upper_rescue['plot_values'])))

    ax.set_xlabel(primary_name)
    ax.set_ylabel(plot_state)
    ax.set_xlim(x_min, float(x_max))
    ax.set_ylim(float(config.get('bistability_plot_ymin', 0.0)), 1.05 * max(ymax_candidates))
    ax.grid(bool(config.get('bistability_plot_grid', False)))
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return output_path


# --- v070 minimal 2D plot override ---
def configure_auto_2d_plot(plotter, experiment, grid=False):
    config = experiment['config']
    plotter.config(
        grid=grid,
        bifurcation_x=[config['secondary_continuation_parameter']],
        bifurcation_y=[config['primary_continuation_parameter']],
        xlabel=config['secondary_continuation_parameter'],
        ylabel=config['primary_continuation_parameter'],
        title='',
    )


# --- v071 robust special-point and generic-plot overrides ---
def maybe_get_label(diagram, label):
    if label is None:
        return None
    try:
        return diagram(label)
    except KeyError:
        return None


def collect_available_labels(diagram):
    labels = []
    for branch in list(getattr(diagram, 'branches', diagram)):
        for point in getattr(branch, 'labels', []):
            label = point.get('label')
            if label is not None:
                labels.append(str(label))
    return labels


def run_one_d_continuation(experiment):
    runner = ensure_runner(experiment)
    config = experiment['config']
    model_name = experiment['model_name']

    eq_forward = ac.run(
        e=model_name,
        c=model_name,
        runner=runner,
        NMX=config['eq_nmx'],
        NPR=config['eq_npr'],
    )
    eq_backward = ac.run(
        DS='-',
        runner=runner,
        NMX=config['eq_nmx'],
        NPR=config['eq_npr'],
    )
    eq_diagram = (eq_forward + eq_backward).relabel()
    ac.save(eq_diagram, 'eq')

    experiment['eq_diagram'] = eq_diagram
    experiment['available_labels'] = collect_available_labels(eq_diagram)
    experiment['bp_seed'] = maybe_get_label(eq_diagram, config.get('branch_point_label'))
    experiment['lp_seed'] = maybe_get_label(eq_diagram, config.get('fold_label'))
    experiment['available_seed_names'] = [
        name
        for name, seed in [('bp_seed', experiment['bp_seed']), ('lp_seed', experiment['lp_seed'])]
        if seed is not None
    ]
    return eq_diagram


def run_two_d_continuation(experiment):
    if 'eq_diagram' not in experiment:
        raise RuntimeError('Run the one-dimensional continuation first.')

    loci = []
    if experiment.get('bp_seed') is not None:
        bp_locus = continue_locus(experiment, experiment['bp_seed'], 'bp_curve')
        experiment['bp_locus'] = bp_locus
        loci.append(bp_locus)

    if experiment.get('lp_seed') is not None:
        lp_locus = continue_locus(experiment, experiment['lp_seed'], 'lp_curve')
        experiment['lp_locus'] = lp_locus
        loci.append(lp_locus)

    if not loci:
        available = ', '.join(experiment.get('available_labels', [])) or 'none'
        raise RuntimeError(
            'No requested codimension-2 seed labels were found in the 1D continuation. '
            f"Requested BP label={experiment['config'].get('branch_point_label')!r}, "
            f"LP label={experiment['config'].get('fold_label')!r}. "
            f'Available labels: {available}.'
        )

    codim2 = loci[0]
    for locus in loci[1:]:
        codim2 = codim2 + locus
    codim2 = codim2.relabel()
    ac.save(codim2, 'codim2')
    experiment['codim2'] = codim2
    return codim2


def save_one_d_plot(experiment, filename=None):
    try:
        return save_backward_s_1d_plot(experiment, filename=filename)
    except Exception:
        output_path = (
            experiment['workdir'] / experiment['one_d_plot_filename']
            if filename is None
            else Path(filename)
        )
        plotter = HeadlessAutoPlotter('eq')
        configure_auto_1d_plot(plotter, experiment, stability=True, grid=False)
        plotter.savefig(output_path)
        return output_path


# --- v072 guarded 1D plot selection override ---
def save_one_d_plot(experiment, filename=None):
    output_path = (
        experiment['workdir'] / experiment['one_d_plot_filename']
        if filename is None
        else Path(filename)
    )

    if experiment.get('lp_seed') is not None:
        try:
            return save_backward_s_1d_plot(experiment, filename=output_path)
        except Exception:
            pass

    plotter = HeadlessAutoPlotter('eq')
    configure_auto_1d_plot(plotter, experiment, stability=True, grid=False)
    plotter.savefig(output_path)
    return output_path


# --- v073 fast special-point lookup override ---
def run_one_d_continuation(experiment):
    runner = ensure_runner(experiment)
    config = experiment['config']
    model_name = experiment['model_name']

    eq_forward = ac.run(
        e=model_name,
        c=model_name,
        runner=runner,
        NMX=config['eq_nmx'],
        NPR=config['eq_npr'],
    )
    eq_backward = ac.run(
        DS='-',
        runner=runner,
        NMX=config['eq_nmx'],
        NPR=config['eq_npr'],
    )
    eq_diagram = (eq_forward + eq_backward).relabel()
    ac.save(eq_diagram, 'eq')

    experiment['eq_diagram'] = eq_diagram
    experiment['bp_seed'] = maybe_get_label(eq_diagram, config.get('branch_point_label'))
    experiment['lp_seed'] = maybe_get_label(eq_diagram, config.get('fold_label'))
    experiment['has_bp_seed'] = experiment['bp_seed'] is not None
    experiment['has_lp_seed'] = experiment['lp_seed'] is not None
    return eq_diagram


def run_two_d_continuation(experiment):
    if 'eq_diagram' not in experiment:
        raise RuntimeError('Run the one-dimensional continuation first.')

    loci = []
    if experiment.get('bp_seed') is not None:
        bp_locus = continue_locus(experiment, experiment['bp_seed'], 'bp_curve')
        experiment['bp_locus'] = bp_locus
        loci.append(bp_locus)

    if experiment.get('lp_seed') is not None:
        lp_locus = continue_locus(experiment, experiment['lp_seed'], 'lp_curve')
        experiment['lp_locus'] = lp_locus
        loci.append(lp_locus)

    if not loci:
        raise RuntimeError(
            'No requested codimension-2 seed labels were found in the 1D continuation. '
            f"Requested BP label={experiment['config'].get('branch_point_label')!r}, "
            f"LP label={experiment['config'].get('fold_label')!r}."
        )

    codim2 = loci[0]
    for locus in loci[1:]:
        codim2 = codim2 + locus
    codim2 = codim2.relabel()
    ac.save(codim2, 'codim2')
    experiment['codim2'] = codim2
    return codim2


In [ ]:

# --- Final output-range overrides for the renamed simulation pair ---

def _headless_savefig_with_limits(self, filename):
    plt = get_matplotlib_pyplot()
    xcolumns = self._resolve_columns(self.options.get('bifurcation_x'))
    ycolumns = self._resolve_columns(self.options.get('bifurcation_y'))
    branches = self._branches()
    fig, ax = plt.subplots(figsize=(6.4, 4.8))
    colors = plt.rcParams['axes.prop_cycle'].by_key().get('color', ['C0'])

    for branch_index, branch in enumerate(branches):
        color = colors[branch_index % len(colors)]
        for xcolumn, ycolumn in zip(xcolumns, ycolumns):
            x = np.asarray(branch.coordarray[xcolumn], dtype=float)
            y = np.asarray(branch.coordarray[ycolumn], dtype=float)

            if self.options.get('stability', False):
                old = 0
                stability = branch.stability()
                if not stability:
                    stability = [len(x)]
                for point in stability:
                    abs_point = abs(int(point))
                    if abs_point > 1 or point == stability[-1]:
                        xs = x[old:abs_point]
                        ys = y[old:abs_point]
                        if len(xs) >= 2:
                            ax.plot(
                                xs,
                                ys,
                                color=color,
                                linestyle='-' if point < 0 else '--',
                                linewidth=1.6,
                            )
                        old = max(abs_point - 1, 0)
            else:
                if len(x) >= 2:
                    ax.plot(x, y, color=color, linewidth=1.6)

    ax.set_xlabel(self.options.get('xlabel', ''))
    ax.set_ylabel(self.options.get('ylabel', ''))
    ax.set_title(self.options.get('title', ''))
    xlim = self.options.get('xlim')
    ylim = self.options.get('ylim')
    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.grid(bool(self.options.get('grid', False)))
    fig.tight_layout()
    fig.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)


HeadlessAutoPlotter.savefig = _headless_savefig_with_limits


def configure_auto_1d_plot(plotter, experiment, stability=True, grid=False):
    config = experiment['config']
    xmax = experiment['primary_max'] if config.get('one_d_plot_max') is None else float(config['one_d_plot_max'])
    xmin = float(config.get('one_d_plot_min', experiment['primary_min']))
    plotter.config(
        stability=stability,
        grid=grid,
        bifurcation_x=[config['primary_continuation_parameter']],
        bifurcation_y=[config['plot_state']],
        xlabel=config['primary_continuation_parameter'],
        ylabel=config['plot_state'],
        title=f"{experiment['model_name']}: 1D bifurcation plot",
        xlim=(xmin, float(xmax)),
    )


def configure_auto_2d_plot(plotter, experiment, grid=False):
    config = experiment['config']
    ymax = float(experiment['primary_max']) if config.get('two_d_plot_ymax') is None else float(config['two_d_plot_ymax'])
    ymin = 0.0 if config.get('two_d_plot_ymin') is None else float(config['two_d_plot_ymin'])
    plotter.config(
        grid=grid,
        bifurcation_x=[config['secondary_continuation_parameter']],
        bifurcation_y=[config['primary_continuation_parameter']],
        xlabel=config['secondary_continuation_parameter'],
        ylabel=config['primary_continuation_parameter'],
        title='',
        xlim=(float(config['secondary_min']), float(config['secondary_max'])),
        ylim=(ymin, ymax),
    )
